In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import numpy as np
import json

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#------------Full 15k Run-------------------------#

#Load our AceReason dataset
from datasets import load_dataset

dataset_url ="redsgnaoh/acereason11_100k"
full_dataset = load_dataset(dataset_url,split="train")

user_questions = full_dataset["instruction"]


#Create embeddings for the user_questions
model = SentenceTransformer("all-MiniLM-L6-v2")

all_embeddings = model.encode(user_questions, batch_size=100,show_progress_bar=True)

np.save("/content/drive/MyDrive/all_embeddings.npy", all_embeddings)


In [ ]:
#create clusters of the user_questions
kmeans_50 = KMeans(n_clusters=50, random_state = 30)
cluster_labels_50 = kmeans_50.fit_predict(all_embeddings)

np.save("/content/drive/MyDrive/cluster_results", cluster_labels_50)


In [ ]:
#create clusters of the user_questions
kmeans_100 = KMeans(n_clusters=100, random_state = 45)
cluster_labels_100 = kmeans_100.fit_predict(all_embeddings)

np.save("/content/drive/MyDrive/100_cluster_results", cluster_labels_100)

In [ ]:
#for cluster size 50

cluster_size_50 = np.bincount(cluster_labels_50)
for c,i in enumerate(cluster_size_50):
  print(c,i)

In [ ]:
#for cluster size 100

cluster_size_100 = np.bincount(cluster_labels_100)
for c,i in enumerate(cluster_size_100):
  print(c,i)

In [ ]:
#for cluster size 50

sample_list = []
rng = np.random.default_rng(2)


#create a random list of samples
for cluster_id in range(kmeans_50.n_clusters):
  sample_cluster_indices = np.where(cluster_labels_50 == cluster_id)[0]
  samples = rng.choice(sample_cluster_indices, size= 15000//kmeans_50.n_clusters,replace=False)

  sample_list.extend(samples)


diversity_dataset = full_dataset.select([int(idx) for idx in sample_list])

with open ("/content/drive/MyDrive/diversity_dataset_50.json","w") as f:
 json.dump(diversity_dataset.to_list(),f)

In [ ]:
#for cluster size 100

sample_list = []
rng = np.random.default_rng(2)


#create a random list of samples
for cluster_id in range(kmeans_100.n_clusters):
  sample_cluster_indices = np.where(cluster_labels_100 == cluster_id)[0]
  samples = rng.choice(sample_cluster_indices, size= 15000//kmeans_100.n_clusters,replace=False)

  sample_list.extend(samples)


diversity_dataset = full_dataset.select([int(idx) for idx in sample_list])

with open ("/content/drive/MyDrive/diversity_dataset_100.json","w") as f:
 json.dump(diversity_dataset.to_list(),f)